# 00 檢查你的 setup

從上到下執行。每個 code cell 最後都應該看到「通過」。若失敗，依訊息打開對應安裝指南補齊後再重跑。

## 1. Course repo path

確認 notebook 正在 `aiops-anomaly-zero-to-hero` repo 內執行。

In [4]:
from pathlib import Path
import os
import platform
import sys


def find_project_root(start: Path) -> Path | None:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "environment.yml").exists() and (candidate / "labs").exists() and (candidate / "infra").exists():
            return candidate
    return None

repo_root = find_project_root(Path.cwd())
if repo_root is None:
    print("FAIL 找不到 aiops-anomaly-zero-to-hero repo。")
    print("請先 cd 到 repo 根目錄，再啟動 JupyterLab 或重新開啟這份 notebook。")
    raise SystemExit("course repo path 檢查失敗")

os.chdir(repo_root)
print("Course repo:", repo_root)
print("Platform:", platform.system(), platform.release(), platform.machine())
print("Python:", sys.executable)
print("通過：notebook 已在此 course repo 中執行。")

Course repo: /Users/kevin/Library/CloudStorage/GoogleDrive-kevin880987@gmail.com/我的雲端硬碟/workshop/workbench/project/202608-aia-network-monitoring-aiops/aiops-anomaly-zero-to-hero
Platform: Darwin 25.5.0 arm64
Python: /opt/miniconda3/envs/aiops-anomaly-zero-to-hero/bin/python
通過：notebook 已在此 course repo 中執行。


## 2. Python kernel / packages

確認目前 notebook kernel 使用課程需要的 Python environment。

In [5]:
import importlib.metadata as metadata
import importlib.util
import platform
import sys

required_packages = {
    "numpy": ("numpy", "1.26"),
    "pandas": ("pandas", "2.0"),
    "scikit-learn": ("sklearn", "1.4"),
    "matplotlib": ("matplotlib", "3.9"),
    "jupyterlab": ("jupyterlab", "4.3"),
    "ipykernel": ("ipykernel", "6.29"),
    "prometheus-client": ("prometheus_client", "0.20"),
}


def version_tuple(value: str) -> tuple[int, ...]:
    parts = []
    for part in value.replace("-", ".").split("."):
        if part.isdigit():
            parts.append(int(part))
            continue
        digits = "".join(ch for ch in part if ch.isdigit())
        if digits:
            parts.append(int(digits))
        break
    return tuple(parts)

failures = []
if sys.version_info < (3, 12):
    failures.append(f"Python >= 3.12 required; current={platform.python_version()}")
else:
    print(f"OK Python {platform.python_version()}")

for package, (module_name, minimum) in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        failures.append(f"missing package: {package}")
        continue
    version = metadata.version(package)
    if version_tuple(version) < version_tuple(minimum):
        failures.append(f"{package} >= {minimum} required; current={version}")
        continue
    print(f"OK {package} {version}")

if failures:
    print("FAIL Python environment 尚未就緒：")
    for item in failures:
        print("-", item)
    print()
    print("請依作業系統打開安裝指南：")
    print("- macOS: labs/getting-started/01a-setup-macos-python-environment.md")
    print("- Linux: labs/getting-started/01b-setup-linux-python-environment.md")
    print("- Windows: labs/getting-started/01c-setup-windows-python-environment.md")
    raise SystemExit("Python kernel / packages 檢查失敗")

print("通過：Python kernel 與必要 packages 已就緒。")

OK Python 3.12.13
OK numpy 2.4.6
OK pandas 3.0.3
OK scikit-learn 1.9.0
OK matplotlib 3.10.9
OK jupyterlab 4.5.8
OK ipykernel 7.3.0
OK prometheus-client 0.25.0
通過：Python kernel 與必要 packages 已就緒。


## 3. 啟動並檢查 course exporter

這一格會檢查 `http://localhost:8000/metrics`。若 exporter 尚未啟動，notebook 會在背景啟動 `infra/rrd_exporter.py`。

In [ ]:
import os
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

EXPORTER_URL = "http://localhost:8000/metrics"
EXPORTER_LOG = Path("outputs/setup/rrd_exporter.log")


def is_reachable(url: str, timeout: float = 2.0) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return 200 <= response.status < 500
    except (urllib.error.URLError, TimeoutError, socket.timeout):
        return False

if is_reachable(EXPORTER_URL):
    print(f"OK course exporter 已在執行: {EXPORTER_URL}")
else:
    print("course exporter 尚未啟動，現在由 notebook 背景啟動。")
    EXPORTER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_file = EXPORTER_LOG.open("a", encoding="utf-8")
    kwargs = {
        "cwd": Path.cwd(),
        "stdout": log_file,
        "stderr": subprocess.STDOUT,
        "env": os.environ.copy(),
    }
    if os.name == "nt":
        kwargs["creationflags"] = subprocess.CREATE_NEW_PROCESS_GROUP
    else:
        kwargs["start_new_session"] = True
    process = subprocess.Popen([sys.executable, "infra/rrd_exporter.py"], **kwargs)

    for _ in range(15):
        time.sleep(1)
        if is_reachable(EXPORTER_URL):
            print(f"OK course exporter 已啟動: {EXPORTER_URL}")
            print(f"Log: {EXPORTER_LOG}")
            break
    else:
        print("FAIL course exporter 無法啟動。")
        print("請查看安裝指南: labs/getting-started/02-install-prometheus.md")
        print(f"Log: {EXPORTER_LOG}")
        if EXPORTER_LOG.exists():
            print()
            print("Log tail:")
            print("".join(EXPORTER_LOG.read_text(encoding="utf-8", errors="replace").splitlines(True)[-20:]))
        raise SystemExit("course exporter 檢查失敗")

print("通過：course exporter 已就緒。")

## 4. Local services

確認 Prometheus、Grafana Local 是否正在執行。`node_exporter` / `windows_exporter` 只有 workshop 即時 OS metrics 需要。

In [6]:
import socket
import urllib.error
import urllib.request

services = [
    ("Prometheus", "http://localhost:9090/-/ready", True, "labs/getting-started/02-install-prometheus.md"),
    ("Grafana Local", "http://localhost:3000/api/health", True, "labs/getting-started/03a-install-grafana-local.md"),
    ("node_exporter", "http://localhost:9100/metrics", False, "labs/getting-started/04-install-node-exporter.md"),
    ("windows_exporter", "http://localhost:9182/metrics", False, "labs/getting-started/04-install-node-exporter.md"),
]

required_failures = []
optional_missing = []
for name, url, required, guide in services:
    try:
        with urllib.request.urlopen(url, timeout=2) as response:
            status = response.status
        ok = 200 <= status < 500
    except (urllib.error.URLError, TimeoutError, socket.timeout):
        ok = False
        status = None

    if ok:
        print(f"OK {name}: {url}")
    elif required:
        print(f"FAIL {name}: 無法連線到 {url}")
        required_failures.append((name, guide))
    else:
        print(f"SKIP {name}: 未啟動。self-study 可略過，workshop 即時 OS metrics 需要。")
        optional_missing.append((name, guide))

if required_failures:
    print()
    print("FAIL 必要服務尚未就緒。請先依下列指南補齊：")
    for name, guide in required_failures:
        print(f"- {name}: {guide}")
    raise SystemExit("Local services 檢查失敗")

print()
print("通過：self-study 必要服務已就緒。")
if optional_missing:
    print("提醒：若要跑 workshop 即時 OS metrics，請再完成：")
    for name, guide in optional_missing:
        print(f"- {name}: {guide}")
else:
    print("通過：workshop 即時 OS metrics exporter 也已就緒。")

FAIL course exporter: 無法連線到 http://localhost:8000/metrics
OK Prometheus: http://localhost:9090/-/ready
OK Grafana Local: http://localhost:3000/api/health
SKIP node_exporter: 未啟動。self-study 可略過，workshop 即時 OS metrics 需要。
SKIP windows_exporter: 未啟動。self-study 可略過，workshop 即時 OS metrics 需要。

FAIL 必要服務尚未就緒。請先依下列指南補齊：
- course exporter: labs/getting-started/02-install-prometheus.md


SystemExit: Local services 檢查失敗

## 5. 可以開始哪一條 lab？

若前面四個 code cell 都通過，且只有 `node_exporter` / `windows_exporter` 被標成 `SKIP`，可以開始：

```text
labs/self-study/00_observability_stack.ipynb
```

若 `node_exporter` 或 `windows_exporter` 也已通過，可以開始 workshop 路線：

```text
labs/workshop/00_observability_stack_and_promql.ipynb
```